# 2026 IEEE Big Data Cup: Traffic Flow Bench
## Task 2: Online Queue-Propagation Forecasting (Shockwave LightGBM)

### Key Highlights:
- **Ground-Truth Mining**: Derives exact binary queue labels ($v \le 0.60 v_f$) from the unmasked training parquets.
- **Shockwave Kinematics**: Computes velocity trends ($\Delta v / \Delta t$), occupancy gradients ($\Delta occ / \Delta t$), and backward shockwave arrival.
- **Calibrated Thresholding**: Directly maximizes space-time IoU ($|Q_{pred} \cap Q_{true}| / |Q_{pred} \cup Q_{true}|$).
- **Covers All 8 Scored Corridors**: Evaluates across both `queue_onset` and `queue_ongoing` conditions.

In [ ]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('../kaggle_public')
OUTPUT_CSV = Path('../datasets/task2_queue_submission.csv')
print('Task 2 environment ready!')

### 1. Inspect Scored Corridors and Forecast Windows

In [ ]:
sample_panel = 'D7_I10_E'
wi_path = DATA_DIR / f'task2/{sample_panel}/validation/window_index.csv'
wi = pd.read_csv(wi_path)
print(f'=== Validation Windows for {sample_panel} ===')
display(wi[['window_id', 'condition', 'forecast_start', 'forecast_end']].head(5))

### 2. Verify Output Predictions & Space-Time Distribution

In [ ]:
if OUTPUT_CSV.exists():
    q_df = pd.read_csv(OUTPUT_CSV)
    print(f'Total Queue Predictions: {len(q_df):,}')
    print('Queue Flag Distribution:')
    print(q_df['queue_pred'].value_counts())
    
    # Plot queue predictions per window
    counts = q_df.groupby('window_id')['queue_pred'].sum().head(10)
    plt.figure(figsize=(12, 4))
    counts.plot(kind='bar', color='crimson')
    plt.title('Predicted Queued Link-Cells per Window')
    plt.ylabel('Count of Queued Cells')
    plt.xticks(rotation=45, ha='right')
    plt.show()
else:
    print('Running Task 2 pipeline...')
    from scripts.build_task2_lightgbm import main as run_task2
    run_task2()